# NYC TLC taxi pickups: street-grid density without a basemap

NYC OpenData's 2015 TLC trip table retains pickup longitude and latitude.
The SODA API returns only those three requested columns, then
XY renders the same rows three ways: automatic scatter density,
explicit hexbin aggregation, and 24 hourly facets. A local projection at
40.75° N corrects longitude scale, so Manhattan's street grid emerges from
pickup points alone without a basemap.

The default reads one million valid pickups in 50,000-row pages.
Increase `TLC_MAX_ROWS` to scale the example; each page is cached so an
interrupted run resumes without repeating completed downloads.

**Source:** [NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)
and the [2015 Yellow Taxi Trip Data table](https://data.cityofnewyork.us/d/2yzn-sicd).
TLC notes that trip records are published as submitted and may contain
inaccuracies.

Install beside XY with `python -m pip install numpy requests xy`.


In [ ]:
import os
import time
from pathlib import Path

import numpy as np
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

import xy

DATA_DIR = Path(os.getenv("XY_REAL_WORLD_DATA", "data")) / "nyc-tlc"
DATA_DIR.mkdir(parents=True, exist_ok=True)

MAX_ROWS = int(os.getenv("TLC_MAX_ROWS", "1000000"))
PAGE_SIZE = int(os.getenv("TLC_PAGE_SIZE", "50000"))
REQUEST_DELAY = float(os.getenv("TLC_REQUEST_DELAY", "0.1"))
if MAX_ROWS <= 0 or not 1 <= PAGE_SIZE <= 50_000 or REQUEST_DELAY < 0:
    raise ValueError("use positive rows, a page size <= 50,000, and a non-negative delay")

SESSION = requests.Session()
SESSION.headers["User-Agent"] = "xy-real-world-notebook/1.0"
app_token = os.getenv("SOCRATA_APP_TOKEN")
if app_token:
    SESSION.headers["X-App-Token"] = app_token
SESSION.mount(
    "https://",
    HTTPAdapter(
        max_retries=Retry(
            total=6,
            backoff_factor=1.0,
            status_forcelist=(429, 500, 502, 503, 504),
            allowed_methods={"GET"},
            respect_retry_after_header=True,
        )
    ),
)

API_URL = "https://data.cityofnewyork.us/resource/2yzn-sicd.csv"
parts = []
for offset in range(0, MAX_ROWS, PAGE_SIZE):
    limit = min(PAGE_SIZE, MAX_ROWS - offset)
    page_path = DATA_DIR / f"pickups-{offset:09d}-{limit:05d}.csv"
    if not page_path.exists():
        response = SESSION.get(
            API_URL,
            params={
                "$select": "pickup_longitude,pickup_latitude,pickup_datetime",
                "$where": (
                    "pickup_longitude between -74.10 and -73.70 "
                    "and pickup_latitude between 40.55 and 40.95"
                ),
                "$order": ":id",
                "$limit": limit,
                "$offset": offset,
            },
            stream=True,
            timeout=(30, 600),
        )
        response.raise_for_status()
        partial = page_path.with_suffix(".csv.part")
        with partial.open("wb") as output:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                output.write(chunk)
        partial.replace(page_path)
        time.sleep(REQUEST_DELAY)

    page = np.loadtxt(
        page_path,
        delimiter=",",
        quotechar='"',
        skiprows=1,
        dtype=[("longitude", "f8"), ("latitude", "f8"), ("pickup_time", "U23")],
        encoding="utf-8",
        ndmin=1,
    )
    if page.size == 0:
        break
    parts.append(page)
    print(f"loaded {sum(part.size for part in parts):,} pickups")
    if page.size < limit:
        break

if not parts:
    raise RuntimeError("NYC OpenData returned no pickup rows")
pickups = np.concatenate(parts)
longitude = pickups["longitude"]
latitude = pickups["latitude"]
pickup_time = pickups["pickup_time"].astype("datetime64[ms]")
hour = (pickup_time.astype("datetime64[h]").astype(np.int64) % 24).astype(np.int16)
print(f"{longitude.size:,} valid pickup locations")

In [ ]:
MAP_LONGITUDE_DOMAIN = (-74.035, -73.765)
MAP_Y_DOMAIN = (40.635, 40.865)
MAP_REFERENCE_LONGITUDE = -73.90
MAP_REFERENCE_LATITUDE = 40.75
MAP_LONGITUDE_SCALE = float(np.cos(np.deg2rad(MAP_REFERENCE_LATITUDE)))
HERO_WIDTH = 1200
HERO_HEIGHT = 700
HERO_PADDING = (116, 168, 76, 148)


def project_longitude(value):
    return (np.asarray(value, dtype=np.float64) - MAP_REFERENCE_LONGITUDE) * MAP_LONGITUDE_SCALE


projected_longitude = project_longitude(longitude)
MAP_X_TICKS = [
    float(project_longitude(value)) for value in [-74.10, -74.00, -73.90, -73.80, -73.70]
]
MAP_X_LABELS = ["74.10° W", "74.00° W", "73.90° W", "73.80° W", "73.70° W"]
MAP_Y_TICKS = [40.65, 40.70, 40.75, 40.80, 40.85]
MAP_Y_LABELS = ["40.65° N", "40.70° N", "40.75° N", "40.80° N", "40.85° N"]

# Derive the hero domain from its exact inner plot dimensions. One projected
# longitude degree therefore occupies the same number of screen pixels as one
# latitude degree, even on the wide gallery canvas.
MAP_PLOT_WIDTH = HERO_WIDTH - HERO_PADDING[1] - HERO_PADDING[3]
MAP_PLOT_HEIGHT = HERO_HEIGHT - HERO_PADDING[0] - HERO_PADDING[2]
MAP_PLOT_ASPECT = MAP_PLOT_WIDTH / MAP_PLOT_HEIGHT
MAP_X_SPAN = (MAP_Y_DOMAIN[1] - MAP_Y_DOMAIN[0]) * MAP_PLOT_ASPECT
MAP_X_DOMAIN = (-MAP_X_SPAN / 2, MAP_X_SPAN / 2)


def nyc_x_axis(*, compact=False):
    return xy.x_axis(
        label=None if compact else "LONGITUDE  /  WEST",
        domain=MAP_X_DOMAIN,
        bounds=MAP_X_DOMAIN,
        tick_values=None if compact else MAP_X_TICKS,
        tick_labels=None if compact else MAP_X_LABELS,
        tick_label_strategy="none" if compact else None,
        style={
            "grid_color": "#193743",
            "grid_width": 1,
            "grid_dash": "dotted",
            "grid_opacity": 0.72,
            "axis_color": "#496979",
            "axis_width": 1,
            "tick_color": "#e6b969",
            "tick_width": 1,
            "tick_length": 4,
            "tick_label_color": "#b9cbd0",
            "label_color": "#f3d59a",
            "tick_label_size": 11,
            "label_size": 12,
        },
    )


def nyc_y_axis(*, compact=False):
    return xy.y_axis(
        label=None if compact else "LATITUDE  /  NORTH",
        label_offset=-18,
        domain=MAP_Y_DOMAIN,
        bounds=MAP_Y_DOMAIN,
        tick_values=None if compact else MAP_Y_TICKS,
        tick_labels=None if compact else MAP_Y_LABELS,
        tick_label_strategy="none" if compact else None,
        style={
            "grid_color": "#193743",
            "grid_width": 1,
            "grid_dash": "dotted",
            "grid_opacity": 0.72,
            "axis_color": "#496979",
            "axis_width": 1,
            "tick_color": "#e6b969",
            "tick_width": 1,
            "tick_length": 4,
            "tick_label_color": "#b9cbd0",
            "label_color": "#f3d59a",
            "tick_label_size": 11,
            "label_size": 12,
        },
    )


def nyc_theme():
    return xy.theme(
        background="#08131b",
        plot_background="#071821",
        text_color="#edf1e8",
        grid_color="#193743",
        axis_color="#496979",
        crosshair_color="#ffca68",
        selection_color="#ffb000",
        selection_fill="#ffb00024",
    )


def nyc_landmarks():
    return (
        xy.callout(
            float(project_longitude(-73.8740)),
            40.7769,
            "LGA  /  QUEENS",
            color="#f3c86d",
            width=1.15,
            dx=22,
            dy=-24,
            style={
                "font_size": 12,
                "font_weight": 700,
                "label_color": "#ffe3a0",
                "letter_spacing": "0.07em",
            },
        ),
        xy.callout(
            float(project_longitude(-73.7781)),
            40.6413,
            "JFK  /  QUEENS",
            color="#f3c86d",
            width=1.15,
            dx=-18,
            dy=-25,
            anchor="end",
            style={
                "font_size": 12,
                "font_weight": 700,
                "label_color": "#ffe3a0",
                "letter_spacing": "0.07em",
            },
        ),
        xy.callout(
            float(project_longitude(-73.9855)),
            40.7580,
            "MIDTOWN CORE\nPICKUP SPINE",
            color="#67d7e5",
            width=1.25,
            dx=-18,
            dy=-64,
            anchor="end",
            style={
                "font_size": 13,
                "font_weight": 700,
                "label_color": "#c8f5f7",
                "letter_spacing": "0.055em",
            },
        ),
    )


def nyc_header(mode):
    header_anchor_y = 1 - (HERO_PADDING[0] - 16) / HERO_HEIGHT
    return (
        xy.text(
            HERO_PADDING[3] / HERO_WIDTH,
            header_anchor_y,
            "NYC NIGHT MOVES",
            dx=0,
            dy=-56,
            color="#fff7e3",
            style={
                "coordinate_space": "figure_fraction",
                "font_size": 26,
                "font_weight": 700,
                "letter_spacing": "0.025em",
            },
        ),
        xy.text(
            HERO_PADDING[3] / HERO_WIDTH,
            header_anchor_y,
            f"{longitude.size:,} PICKUPS  ·  YELLOW CAB 2015  ·  {mode}",
            dx=0,
            dy=-12,
            color="#7fa8b2",
            style={
                "coordinate_space": "figure_fraction",
                "font_size": 10,
                "font_weight": 700,
                "letter_spacing": "0.09em",
            },
        ),
    )


NYC_CHROME_STYLES = {
    "title": {
        "text_align": "left",
        "font_size": 22,
        "font_weight": 700,
        "letter_spacing": "0.055em",
    },
    "tick_label": {"font_family": "Avenir Next, ui-sans-serif, sans-serif"},
    "axis_title": {
        "font_family": "Avenir Next, ui-sans-serif, sans-serif",
        "letter_spacing": "0.08em",
    },
    "annotation_label": {
        "font_family": "Avenir Next, ui-sans-serif, sans-serif",
        "letter_spacing": "0.08em",
    },
    "colorbar": {
        "background": "#0d202a",
        "color": "#f3d59a",
        "border": "1px solid #294652",
        "border_radius": 5,
        "padding": "8px 7px",
    },
    "colorbar_title": {"font_size": 11, "font_weight": 700, "letter_spacing": "0.055em"},
    "colorbar_tick": {"font_size": 11},
    "tooltip": {
        "background": "#0d202a",
        "color": "#edf1e8",
        "border": "1px solid #496979",
        "border_radius": 5,
        "font_family": "Avenir Next, ui-sans-serif, sans-serif",
    },
}

density_chart = xy.scatter_chart(
    xy.scatter(
        projected_longitude,
        latitude,
        name="pickup intensity",
        color="#ffd166",
        size=1.1,
        opacity=1.0,
        density=True,
    ),
    xy.scatter(
        projected_longitude[:: max(1, longitude.size // 75_000)],
        latitude[:: max(1, latitude.size // 75_000)],
        name="pickup detail",
        color="#ffe7a3",
        size=0.7,
        opacity=0.16,
        density=False,
    ),
    *nyc_landmarks(),
    *nyc_header("ALL-HOUR  ·  ADAPTIVE SCATTER DENSITY"),
    nyc_x_axis(),
    nyc_y_axis(),
    xy.tooltip(title="YELLOW-TAXI PICKUP"),
    xy.legend(show=False),
    xy.interaction_config(crosshair=True, wheel_zoom=True, box_zoom=True),
    nyc_theme(),
    styles=NYC_CHROME_STYLES,
    style={
        "border": "1px solid #294652",
        "font_family": "Avenir Next, ui-sans-serif, sans-serif",
    },
    width=HERO_WIDTH,
    height=HERO_HEIGHT,
    padding=HERO_PADDING,
)
print(
    "scatter tier:",
    density_chart.figure().build_payload()[0]["traces"][0]["tier"],
)
density_chart

In [ ]:
hexbin_chart = xy.hexbin_chart(
    xy.hexbin(
        projected_longitude,
        latitude,
        name="pickups per hex",
        gridsize=(210, 260),
        mincnt=2,
        bins="log",
        colormap="magma",
        opacity=0.96,
    ),
    *nyc_landmarks(),
    *nyc_header("ALL-HOUR  ·  LN-SCALED HEX DENSITY"),
    nyc_x_axis(),
    nyc_y_axis(),
    xy.colorbar(
        title="LN(1 + PICKUPS / HEX)",
        ticks=[2, 3, 4, 5, 6, 7],
        orientation="vertical",
    ),
    xy.tooltip(title="LN(1 + PICKUPS / HEX)", format={"value": ".2f"}),
    xy.legend(show=False),
    xy.interaction_config(crosshair=True, wheel_zoom=True, box_zoom=True),
    nyc_theme(),
    styles=NYC_CHROME_STYLES,
    style={
        "border": "1px solid #294652",
        "font_family": "Avenir Next, ui-sans-serif, sans-serif",
    },
    width=HERO_WIDTH,
    height=HERO_HEIGHT,
    padding=HERO_PADDING,
)
hexbin_chart

In [ ]:
hour_order = np.argsort(hour, kind="stable")
hour_labels = np.array([f"{value:02d}:00" for value in range(24)])[hour[hour_order]]
hourly_data = {
    "map_x": projected_longitude[hour_order],
    "latitude": latitude[hour_order],
    "local_hour": hour_labels,
}
hourly_chart = xy.facet_chart(
    xy.scatter(
        x="map_x",
        y="latitude",
        name="pickup intensity",
        color="#ffd166",
        size=1.0,
        opacity=1.0,
        density=True,
    ),
    xy.scatter(
        x="map_x",
        y="latitude",
        name="pickup detail",
        color="#ffe7a3",
        size=0.65,
        opacity=0.18,
        density=False,
    ),
    nyc_x_axis(compact=True),
    nyc_y_axis(compact=True),
    xy.legend(show=False),
    nyc_theme(),
    by="local_hour",
    data=hourly_data,
    cols=6,
    share_x=True,
    share_y=True,
    title="NYC NIGHT MOVES  ·  PICKUP DENSITY BY LOCAL HOUR",
    styles={
        **NYC_CHROME_STYLES,
        "title": {
            "font_size": 11,
            "font_weight": 700,
            "letter_spacing": "0.08em",
        },
    },
    style={
        "border": "1px solid #294652",
        "font_family": "Avenir Next, ui-sans-serif, sans-serif",
    },
    width=1320,
    height=210,
    padding=(30, 16, 16, 16),
    gap=0,
)
hourly_chart